## Introduction

Home Credit is an international consumer finance company. The company provides various services in emerging markets with a focus on customers with limited credit histories. They want to broaden financial inclusion for the unbanked (underserved) population with insufficient or non-existent credit histories. They believe this can be achieved by making use of a variety of alternative data: telco, transactional or other.

## Goal

Predict how capable each applicant is of repaying a loan, based on the provided dataset.

- **TARGET = 1**: the client had payment difficulties (late on at least one of the first installments)
- **TARGET = 0**: all other cases
- **Metric:** ROC AUC. Only about 8% of clients default, so accuracy would be misleading.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

DATA = Path("..") / "data"

## 1. Load and first look

In [ ]:
df = pd.read_csv(DATA / "application_train.csv")
df.shape

In [ ]:
df.head()

In [ ]:
df.info(memory_usage="deep")

In [ ]:
df["SK_ID_CURR"].is_unique

One row = one loan application, no duplicates. `SK_ID_CURR` is the key for joining the other tables.

## 2. Target and the accuracy trap

In [ ]:
df["TARGET"].value_counts(normalize=True)

In [ ]:
y = df["TARGET"]
lazy = np.zeros(len(y))

accuracy_score(y, lazy), roc_auc_score(y, lazy)

A model that predicts "repays" for everyone is 92% accurate but has AUC 0.5: every score ties, so it cannot rank defaulters above good clients. Accuracy is misleading at an 8% default rate, so we use ROC AUC.

## 3. Missing values

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
(missing > 0).sum()

In [ ]:
df_description = pd.read_csv(DATA / "HomeCredit_columns_description.csv", encoding="latin-1")

desc = (
    df_description[df_description["Table"] == "application_{train|test}.csv"]
    .set_index("Row")["Description"]
)

top_missing = missing.head(20).to_frame("missing_share").join(desc)
top_missing

In [ ]:
df.groupby(df["EXT_SOURCE_1"].isna())["TARGET"].mean()

- The most-missing columns describe the client's building (`_AVG`, `_MODE`, `_MEDI`: three summaries of the same facts) and `OWN_CAR_AGE` (missing mostly because the client has no car).
- Clients without `EXT_SOURCE_1` default about 14% more often (8.5% vs 7.5%), so missingness itself carries signal.
- Don't drop columns by a missing-share threshold: a column can be highly predictive where it exists.

## 4. Column types

In [ ]:
df.dtypes.value_counts()

In [ ]:
cat_cols = df.select_dtypes("object").columns
df[cat_cols].nunique().sort_values(ascending=False)

In [ ]:
df["ORGANIZATION_TYPE"].value_counts().head(10)

In [ ]:
df[["DAYS_BIRTH", "DAYS_EMPLOYED"]].describe()

_Findings: to fill in after Lesson 4._